# Accelerating KNN with parallel computing


In [56]:
pip install dask

Note: you may need to restart the kernel to use updated packages.


In [57]:
import numpy as np
import time
from sklearn.datasets import make_classification
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score
import multiprocessing
from joblib import Parallel, delayed
import dask
import dask.array as da

#### **Task 1: serial implementation**
**Goal**: Create a dataset and run KNN the normal way (one by one) to see how long it takes.

##### **1.1 Creating the Data**
I will create a dataset with 20,000 samples and 20 features. I will keep 500 samples for testing.

In [58]:
# Generate a synthetic dataset
X, y = make_classification(n_samples=20000, n_features=20, n_informative=15, n_classes=2, random_state=42)

# Split data into training and testing sets
# We want exactly 500 test points
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=500, random_state=42)

print(f"Training data shape: {X_train.shape}")
print(f"Testing data shape: {X_test.shape}")

Training data shape: (19500, 20)
Testing data shape: (500, 20)


##### **1.2 The KNN function**
Here is the function that calculates the distance. It looks at one test point and compares it to all training points to find the closest neighbors.

In [59]:
def knn_predict_single(test_point, X_train, y_train, k=5):
    """
    Predicts the label for one point.
    It finds the 'k' closest points in the training set.
    """
    # 1. Calculate the distance to all training points
    # We use NumPy to do this fast (vectorization)
    distances = np.linalg.norm(X_train - test_point, axis=1)
    
    # 2. Find the indices of the k smallest distances
    k_indices = distances.argsort()[:k]
    
    # 3. Get the labels of those neighbors
    k_nearest_labels = y_train[k_indices]
    
    # 4. Vote: count which label appears most often
    most_common = np.bincount(k_nearest_labels).argmax()
    
    return most_common

##### **1.3 Running it serially**
Now I run the function for every test point, one after another, and measure the time.

In [60]:
def run_serial_knn(X_test, X_train, y_train, k=5):
    start_time = time.time()
    # This list runs one by one
    predictions = [knn_predict_single(x, X_train, y_train, k) for x in X_test]
    end_time = time.time()
    
    duration = end_time - start_time
    accuracy = accuracy_score(y_test, predictions)
    
    return predictions, duration, accuracy

# Run the serial version
pred_serial, time_serial, acc_serial = run_serial_knn(X_test, X_train, y_train)

print(f"--- Task 1: Serial (Normal) Execution ---")
print(f"Time taken: {time_serial:.4f} seconds")
print(f"Accuracy: {acc_serial:.4f}")

--- Task 1: Serial (Normal) Execution ---
Time taken: 1.1500 seconds
Accuracy: 0.9760


#### **Task 2: manual parallelism (multiprocessing)**
**Goal**: Use all the CPU cores at the same time to finish faster.

I will use the multiprocessing module. This creates separate "workers" that can work on different parts of the data at the same time.

In [61]:

# Create synthetic dataset
X, y = make_classification(
    n_samples=500, n_features=20, n_informative=15, n_classes=2, random_state=42
)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)


# Multiprocessing version
def run_multiprocessing_knn(X_test, X_train, y_train, k=5):
    num_cores = multiprocessing.cpu_count()
    start_time = time.time()
    
    # Pack arguments for each test sample
    args_list = [(x, X_train, y_train, k) for x in X_test]
    
    with multiprocessing.Pool(processes=num_cores) as pool:
        predictions = pool.map(knn_worker, args_list)
    
    duration = time.time() - start_time
    accuracy = accuracy_score(y_test, predictions)
    
    return predictions, duration, accuracy


# Run multiprocessing KNN
pred_mp, time_mp, acc_mp = run_multiprocessing_knn(X_test, X_train, y_train)
print("\n--- Task 2: Multiprocessing KNN ---")
print(f"Time taken: {time_mp:.4f} seconds")
print(f"Speedup compared to serial: {time_serial / time_mp:.2f}x faster")
print(f"Accuracy: {acc_mp:.4f}")


--- Task 2: Multiprocessing KNN ---
Time taken: 0.1346 seconds
Speedup compared to serial: 8.54x faster
Accuracy: 0.9200


#### **Task 3: production-grade scaling (Joblib & Dask)**
##### **3.1 using Joblib**
**Goal**: Use a library made for science (Joblib) to make the code simpler.

Joblib makes it very easy to run things in parallel without managing the workers manually.

In [62]:
def run_joblib_knn(X_test, X_train, y_train, k=5):
    start_time = time.time()
    
    # Parallel runs the function in parallel
    # n_jobs=-1 means use all available cores
    predictions = Parallel(n_jobs=-1)(
        delayed(knn_predict_single)(x, X_train, y_train, k) for x in X_test
    )
    
    end_time = time.time()
    duration = end_time - start_time
    accuracy = accuracy_score(y_test, predictions)
    
    return predictions, duration, accuracy

# Run Joblib version
pred_joblib, time_joblib, acc_joblib = run_joblib_knn(X_test, X_train, y_train)

print(f"--- Task 3a: Joblib ---")
print(f"Time taken: {time_joblib:.4f} seconds")
print(f"Speedup: {time_serial / time_joblib:.2f}x faster")
print(f"Accuracy: {acc_joblib:.4f}")

--- Task 3a: Joblib ---
Time taken: 0.0842 seconds
Speedup: 13.66x faster
Accuracy: 0.9200


##### **3.2 using Dask**
**Goal**: Use Dask to handle the tasks.

Dask is good for big data. Here, I use dask.delayed to tell Dask to build a plan of what to do, and then run everything in parallel.

In [63]:
def run_dask_knn(X_test, X_train, y_train, k=5):
    # Make the function "lazy" (it waits to run)
    lazy_knn = dask.delayed(knn_predict_single)
    
    # Create a list of tasks to do
    lazy_predictions = [lazy_knn(x, X_train, y_train, k) for x in X_test]
    
    start_time = time.time()
    
    # Compute actually runs the tasks in parallel
    predictions = dask.compute(*lazy_predictions)
    
    end_time = time.time()
    duration = end_time - start_time
    accuracy = accuracy_score(y_test, predictions)
    
    return predictions, duration, accuracy

# Run Dask version
pred_dask, time_dask, acc_dask = run_dask_knn(X_test, X_train, y_train)

print(f"--- Task 3b: Dask (Delayed) ---")
print(f"Time taken: {time_dask:.4f} seconds")
print(f"Speedup: {time_serial / time_dask:.2f}x faster")
print(f"Accuracy: {acc_dask:.4f}")

--- Task 3b: Dask (Delayed) ---
Time taken: 0.0442 seconds
Speedup: 26.01x faster
Accuracy: 0.9200


#### **Task 4: critical thinking**


##### **1. Overhead**
**Question: Why is parallel slower if we only classify 5 test points?**
**Answer**: If we only classify a very small number of points, like 5, the parallel version will likely be slower. This is because of communication overhead.

-Setup Time: It takes time for the computer to start the worker processes.  
-Data Transfer: The data must be "serialized" (converted to a format to send over the network) and sent to the workers.  
-Result: For 5 points, the time spent setting up and sending data is actually longer than the time spent doing the math. Therefore, the serial version is faster for tiny tasks.

##### **2. Memory Constraints**
**Question: If the training data was 200GB, which method would I use?**
**Answer**: I would use Dask.

-The Problem: Standard Python methods (Serial, Multiprocessing, and Joblib) try to load the entire dataset into the computer's RAM (memory) at once. If the dataset is 200GB, most computers will run out of memory and crash.  
-The Solution: Dask is designed for "Big Data." It uses a technique called Out-of-Core computing. This means Dask breaks the huge 200GB file into small "chunks" that fit in memory. It processes these chunks one by one. This allows us to analyze data that is much larger than our computer's RAM.

##### **3. The GIL**
**Question: Why do we use multiprocessing instead of threading for KNN?**
**Answer**: We use multiprocessing instead of threading because of the Global Interpreter Lock (GIL).

-What is the GIL? In Python, the GIL is a lock that allows only one thread to execute Python code at a time.  
-Why it matters here: KNN is a CPU-intensive task (it does a lot of mathematical calculations). If we used threading, the threads would constantly fight for the GIL, and they could not run at the same time.  
-The Solution: Multiprocessing creates separate processes with their own memory space. This bypasses the GIL and allows all CPU cores to do the math simultaneously.